In [50]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler , OrdinalEncoder , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator , TransformerMixin

In [40]:
'''                         data_set 
                            /      \ 
                        num         cat
                         |           |
                imputer(median)     imputer(most_frequent)
                         |           |
                         |          ord_encoding(imp_cat_cols)
                         |           |
                         |          one_hot_encoding(Gender , School_Type)(optional)
                         |           |
                column_adder(num)   column_adder(all)    
                         |           |
                standard_scaler      |   
                          \          /
                           \        /
                            \      /
                             \    / 
                              \  /
                              fit
'''

'                         data_set \n                            /      \\ \n                        num         cat\n                         |           |\n                imputer(median)     imputer(most_frequent)\n                         |           |\n                         |          ord_encoding(imp_cat_cols)\n                         |           |\n                         |          one_hot_encoding(Gender , School_Type)(optional)\n                         |           |\n                column_adder(num)   column_adder(all)    \n                         |           |\n                standard_scaler      |   \n                          \\          /\n                           \\        /\n                            \\      /\n                             \\    / \n                              \\  /\n                              fit\n'

In [41]:
X_train = pd.read_csv('Datasets\\student_data\\Student_X_train.csv')
y_train = pd.read_csv('Datasets\\student_data\\Student_y_train.csv')
test    = pd.read_csv('Datasets\\student_data\\Student_test.csv')

In [42]:
def Multiply_Ordinal_All(cols , data):
    data['All'] = np.ones(len(data))
    for i in cols:
        data['All'] = data['All']*(data[f'ord__{i}'] + 1)

        
    

In [43]:


class Columns_Attribs_Adder(BaseEstimator , TransformerMixin):

    def __init__(self , all=True ):
        self.all = all
        

    def fit(self):
        return self
    
    def tranform(self , X , imp_cols):

        X['study-sleep']    = X['Hours_Studied'] - X['Sleep_Hours']
        X['att*study_hrs']  = X['Attendance']*X['Hours_Studied']
        X['prev*study_hrs'] = X['Previous_Scores']*X['Hours_Studied']
        X['tutor*study_hrs']= X['Tutoring_Sessions']*X['Hours_Studied']

        if self.all :

            Multiply_Ordinal_All(data = X , cols = imp_cols)







In [44]:


num_attribs = [col for col in X_train.columns if X_train[col].dtype != 'object']
cat_attribs = [col for col in X_train.columns if X_train[col].dtype == 'object']

ordinal_cols = ['Teacher_Quality','Family_Income','Access_to_Resources' ,'Parental_Involvement',
                'Motivation_Level','Peer_Influence' ,'Parental_Education_Level',
                'Distance_from_Home']

ohe_cols  = list(set(cat_attribs) - set(ordinal_cols))

all_cols = list(X_train.columns)

imp_cat_col = ('Teacher_Quality', 'Access_to_Resources', 'Parental_Involvement', 
               'Peer_Influence')

ord_categories = [ ['Low','Medium','High'],
             ['Low','Medium','High'],
             ['Low','Medium','High'],
             ['Low','Medium','High'],
             [ 'Low','Medium','High'],
             ['Negative','Neutral','Positive'],
             ['High School','College','Postgraduate'],

             ['Near', 'Moderate', 'Far']
             ]







In [45]:
def custom_pipeline(X , cat_attribs=cat_attribs,  
                    num_attribs=num_attribs  , 
                    ohe_cols = ohe_cols,
                    ordinal_cols = ordinal_cols , 
                    imp_cat_col = imp_cat_col , categories = ord_categories):
    
    ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=ord_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
    ])

    ohe_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore'
        ))
    ])

    full_pipeline = ColumnTransformer([
        ('ord', ordinal_pipeline, ordinal_cols),
        ('ohe', ohe_pipeline, ohe_cols)
    ], remainder='passthrough')


    num_pipeline = Pipeline(
    [('impute_median' , SimpleImputer(strategy='median') ),
    ('scaler' , StandardScaler())]
    )

    full_pipeline.set_output(transform="pandas")
    X_tr = full_pipeline.fit_transform(X)

    add_attribs = Columns_Attribs_Adder()
    X_tr.columns = X_tr.columns.str.replace('remainder__' ,'')
    add_attribs.tranform(X=X_tr , imp_cols= imp_cat_col)

    num_full_pipeline = ColumnTransformer(transformers=[
    ('num' , num_pipeline , num_attribs)
    ])
    
    X_tr = num_full_pipeline.fit_transform(X_tr)
    
    return X_tr
    

In [47]:

num_pipeline = Pipeline(
    [('impute_median' , SimpleImputer(strategy='median') ),
    ('scaler' , StandardScaler())]
    )

full_pipeline = ColumnTransformer(transformers = [
   ('most_freq' , SimpleImputer(strategy='most_frequent') , cat_attribs),
   ('ord' , OrdinalEncoder(categories=ord_categories) , ordinal_cols ),
   ('ohe' , OneHotEncoder(sparse_output=False) , ohe_cols) ,
],remainder='passthrough')
       

full_pipeline.set_output(transform="pandas")

add_attribs = Columns_Attribs_Adder()



X_tr = full_pipeline.fit_transform(X_train)
X_tr.columns = X_tr.columns.str.replace('remainder__' ,'')
add_attribs.tranform(X=X_tr , imp_cols = imp_cat_col)

num_full_pipeline = ColumnTransformer(transformers=[
    ('num' , num_pipeline , num_attribs)
])

num_full_pipeline.fit_transform(X_tr)

array([[-6.60524884e-01, -1.46920241e+00,  6.66829092e-01, ...,
         2.95969389e-02, -1.26660045e+00, -6.49492418e-02],
       [-4.93848022e-01,  6.91028640e-01,  6.66829092e-01, ...,
        -9.41770920e-01,  4.83815863e-01, -6.49492418e-02],
       [ 3.39536287e-01, -2.45294971e-04, -2.04722823e+00, ...,
         2.95969389e-02, -3.91392292e-01, -6.49492418e-02],
       ...,
       [ 6.72890011e-01, -2.59473021e-01,  6.66829092e-01, ...,
        -9.41770920e-01, -3.91392292e-01, -6.49492418e-02],
       [-1.66058605e+00,  1.03666561e+00, -6.90199570e-01, ...,
        -9.41770920e-01,  1.35902402e+00, -1.57017641e+00],
       [-8.27201746e-01,  6.04619398e-01,  2.02385775e+00, ...,
         1.00096480e+00,  4.83815863e-01, -6.49492418e-02]])

In [48]:
X_for_model = custom_pipeline(X_train  ,cat_attribs=cat_attribs,  
                    num_attribs=num_attribs  , 
                    ohe_cols = ohe_cols,
                    ordinal_cols = ordinal_cols , 
                    imp_cat_col = imp_cat_col , categories = ord_categories)


In [49]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error 
from sklearn.model_selection import cross_val_score
lin_reg = LinearRegression()
lin_reg.fit(X_for_model , y_train)

pred = lin_reg.predict(X_for_model)
lin_mse = mean_squared_error(y_train , pred)
lin_rmse = np.sqrt(lin_mse)
lin_rmse


2.4909037161961147

In [51]:
scores = cross_val_score(lin_reg, X_for_model, y_train,
 scoring="neg_mean_squared_error", cv=10)

cv_rmse = np.sqrt(-scores)
cv_rmse.mean()


2.4659822099665765

In [52]:
X_test = test.drop(columns='Exam_Score') 
y_test = test['Exam_Score'] 
X_test_model =  custom_pipeline( X_test ,cat_attribs=cat_attribs,  
                    num_attribs=num_attribs  , 
                    ohe_cols = ohe_cols,
                    ordinal_cols = ordinal_cols , 
                    imp_cat_col = imp_cat_col)

test_pred = lin_reg.predict(X_test_model)
lin_mse = mean_squared_error(y_test, test_pred)
lin_rmse = np.sqrt(lin_mse)
lin_rmse



2.253998217752278

In [46]:
scores = cross_val_score(lin_reg, X_test_model, y_test,
 scoring="neg_mean_squared_error", cv=10)


cv_rmse = np.sqrt(-scores)
cv_rmse.mean()


2.221940870189467

# Tree Reg

In [57]:
from sklearn.tree import DecisionTreeRegressor
tree_reg =  DecisionTreeRegressor()
tree_reg.fit(X_for_model ,y_train)
y_tree_pred = tree_reg.predict(X_for_model)
tree_mse = mean_squared_error(y_train , y_tree_pred)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

0.05754897858046489

In [64]:
tree_score  = cross_val_score(tree_reg,X_for_model,y_train,
                              scoring='neg_mean_squared_error', cv =  10)
tree_score_rmse_mean = np.sqrt(-tree_score).mean()
tree_score_rmse_mean

3.8067478218733384

In [61]:
from sklearn.ensemble import RandomForestRegressor
forest_reg =  RandomForestRegressor()
forest_reg.fit(X_for_model ,y_train)
y_forest_pred = forest_reg.predict(X_for_model)
forest_mse = mean_squared_error(y_train , y_forest_pred)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

d:\Python\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


1.022764008333826

In [63]:
forest_score  = cross_val_score(forest_reg,X_for_model,y_train,
                              scoring='neg_mean_squared_error', cv =  10)
forest_score_rmse_mean = np.sqrt(-forest_score).mean()
forest_score_rmse_mean

d:\Python\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
d:\Python\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
d:\Python\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
d:\Python\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args

2.700738767901702

In [66]:
import joblib


joblib.dump(lin_reg, "student_model.pkl")

['student_model.pkl']